# Macro Surprise + VWAP Execution

Ce notebook est la synthese lisible de l'etude. L'objectif est de montrer ce qui a ete teste, ce qui a change entre V1 et V2, et pourquoi certains noms sont retenus ou rejetes.

Je garde une lecture prudente : ce n'est pas encore une strategie prete a trader. C'est une recherche sur les jours de publications macro, combinee a une execution intraday autour du VWAP.

## Idee de recherche

Je teste si les surprises macro US peuvent aider a identifier des patterns court terme sur certaines actions.

Le signal macro vient de la difference entre la valeur publiee et le consensus. Ensuite, le VWAP sert de filtre d'execution : pour un long, le modele prefere entrer sous ou proche du VWAP ; pour un short, il prefere entrer au-dessus ou proche du VWAP.

Point important : beaucoup de publications US sortent avant l'ouverture du marche actions. Donc je prefere decrire ce travail comme :

**macro calendar day + VWAP intraday**

et pas comme une reaction pure minute par minute a la publication.

## Signal macro et hypotheses

Formule simple :

```text
surprise_raw = actual - estimate
surprise_z = (surprise_raw - historical_average_surprise) / historical_surprise_std
signal = surprise_z * direction
```

`direction = +1` quand une valeur plus haute est consideree comme positive.

`direction = -1` quand une valeur plus basse est consideree comme positive.

Le moteur teste ensuite :

- `DRIFT` : trader dans le sens du signal macro.
- `FADE` : trader contre le signal macro.

Les evenements `MIXED` sont exclus.

## Ce qui change entre V1 et V2

V1 etait le premier screening US macro + VWAP.

V2 rend la validation train/test plus stricte et plus lisible. Le point cle est de separer :

- le rendement du meilleur backtest full-sample ;
- la performance de la regle selectionnee sur le train ;
- la performance de cette meme regle sur la periode test.

Si une action a un bon full-sample mais que la regle selectionnee sur le train perd de l'argent sur le test, je prefere la rejeter. C'est souvent un signe d'overfit ou de regime trop specifique.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display, Image, Markdown
except ModuleNotFoundError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image({self.filename})"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "macro-vwap-event-study" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

def load_csv(path):
    path = Path(path)
    if path.exists():
        df = pd.read_csv(path)
        print(f"Charge {path.name}: {df.shape[0]} lignes, {df.shape[1]} colonnes")
        return df
    print(f"AVERTISSEMENT: fichier manquant: {path}")
    return None

def display_ticker(ticker):
    if str(ticker).upper() == "LONDON-STRATEGIC-EDGE":
        return "LMB"
    return str(ticker)

def add_display_ticker(df):
    if df is not None and "ticker" in df.columns:
        df = df.copy()
        df["display_ticker"] = df["ticker"].map(display_ticker)
    return df

def show_figures(fig_dir):
    fig_dir = Path(fig_dir)
    pngs = sorted(fig_dir.glob("*.png"))
    if not pngs:
        display(Markdown(f"Aucune figure PNG trouvee dans `{fig_dir}`."))
        return
    for path in pngs:
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path)))

V1_DIR = PROJECT_ROOT / "research_tests" / "test_01_us_macro_vwap_screening" / "results"
V2_DIR = PROJECT_ROOT / "research_tests" / "test_02_train_test_validation" / "results"

v1_master = add_display_ticker(load_csv(V1_DIR / "csv_outputs" / "single_country_master_summary.csv"))
v2_master = add_display_ticker(load_csv(V2_DIR / "csv_outputs" / "single_country_master_summary.csv"))
v2_train_test = add_display_ticker(load_csv(V2_DIR / "csv_outputs" / "single_country_train_test.csv"))
v2_walk = add_display_ticker(load_csv(V2_DIR / "csv_outputs" / "single_country_walk_forward.csv"))

## Controle des tickers

Je ne modifie pas les CSV bruts, mais j'ajoute une colonne de presentation dans le notebook.

Si `LONDON-STRATEGIC-EDGE` apparait, je l'affiche comme LMB car le fichier source correspond a `lmb_dataset_London-Strategic-Edge.csv`.

Je signale aussi les doublons, surtout IRTC, car ce n'est pas quelque chose a cacher dans une revue.

In [ ]:
for label, df in [("V1", v1_master), ("V2", v2_master)]:
    if df is None or "ticker" not in df.columns:
        continue
    print(f"\n{label}")
    bad = df[df["ticker"].astype(str).str.upper().eq("LONDON-STRATEGIC-EDGE")]
    if not bad.empty:
        display(Markdown("Ticker naming issue: `LONDON-STRATEGIC-EDGE` est affiche comme `LMB` dans les tableaux."))
        display(bad[["ticker", "display_ticker", "price_file"]] if "price_file" in bad.columns else bad[["ticker", "display_ticker"]])
    dupes = df[df["ticker"].duplicated(keep=False)]
    if not dupes.empty:
        display(Markdown("Tickers dupliques detectes :"))
        display(dupes[["ticker", "display_ticker", "price_file"]] if "price_file" in dupes.columns else dupes[["ticker", "display_ticker"]])

## Resultats V2 en synthese

La V2 est plus selective que la V1.

Lecture actuelle :

- A-class : SMTC, VIAV
- B-class : RVLV, SAM
- Watchlist / C : REAL, IRTC, LMB, PLAY, VSAT, BLFS, GPRO
- Rejetes : BOOT, ENVX, KOP, LMND, LUMN, VREX

Le point important est que le framework ne prend pas juste les meilleurs rendements full-sample. Il regarde si la logique tient aussi sur la periode test.

In [ ]:
if v2_master is not None:
    counts = v2_master["classification"].value_counts().rename_axis("classification").reset_index(name="count")
    display(Markdown("**Mix de classifications V2**"))
    display(counts)

    cols = [
        "display_ticker", "classification", "confidence_score", "test_status",
        "best_hypothese", "best_vwap_mode", "best_horizon",
        "best_ret_moy_pct", "train_ret_moy_pct", "test_ret_moy_pct",
        "random_side_percentile", "random_ts_percentile", "cost_break_even_bps", "main_warning"
    ]
    cols = [c for c in cols if c in v2_master.columns]
    ranked = v2_master.sort_values(["confidence_score", "test_ret_moy_pct"], ascending=False)
    display(Markdown("**Top V2 par confidence_score**"))
    display(ranked[cols].head(15))

## Figures V2

Les figures ci-dessous sont les figures du run V2. Elles ont ete regenerees depuis les CSV V2 car le dossier de resultats V2 fourni ne contenait pas de PNG.

In [ ]:
show_figures(V2_DIR / "figures")

## Pourquoi certains noms profitables sont rejetes

Un rendement full-sample positif ne suffit pas.

Je veux savoir si la regle choisie sur la periode ancienne continue de marcher sur la periode plus recente. Si ce n'est pas le cas, je traite le resultat comme probablement overfit ou dependant d'un regime precis.

In [ ]:
if v2_master is not None:
    rejected_profitable = v2_master[
        (pd.to_numeric(v2_master["best_ret_moy_pct"], errors="coerce") > 0)
        & (pd.to_numeric(v2_master["test_ret_moy_pct"], errors="coerce") <= 0)
    ].copy()
    cols = ["display_ticker", "best_ret_moy_pct", "train_ret_moy_pct", "test_ret_moy_pct", "classification", "test_status", "main_warning"]
    cols = [c for c in cols if c in rejected_profitable.columns]
    display(rejected_profitable[cols].sort_values("best_ret_moy_pct", ascending=False))

## Conclusion actuelle

SMTC et VIAV sont les candidats les plus propres en V2. RVLV et SAM restent interessants, mais je les lis comme plus fragiles parce que la validation train/test est moins nette.

REAL et IRTC restent utiles en watchlist, mais le signal semble plus VWAP-driven ou moins clairement lie a la direction macro.

La prochaine etape doit etre une V3 plus concentree sur quelques noms, avec plus de simulations random, une comparaison plus stricte entre `close_cross` et `limit_touch`, et une lecture par annee / type d'evenement macro.